In [1]:
import numpy as np
import pandas as pd




# Get data

In [2]:
hist = pd.read_csv("/Users/macbookpro/platform/Backend/data/processed/default_hist.csv")
orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')



/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_2382/2619987728.py:2: DtypeWarning: Columns (24,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')


In [3]:
loan_hist = hist[hist['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']
loan_orig = orig[orig['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']


# Config

In [4]:
train_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_train.yaml'
test_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_test.yaml'

# Feature and scaler

In [5]:
import importlib
import src.LGDcomponent.pipelines.lgdFeaturePipeline as lgdFeaturePipeline
importlib.reload(lgdFeaturePipeline)
from src.LGDcomponent.pipelines.lgdFeaturePipeline import LGDFeaturePipeline

/Users/macbookpro/platform/Backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pipeline = LGDFeaturePipeline(config_path=train_cfPath)

scaler config loaded successfully


In [12]:
X,y = pipeline.build(hist, orig)

Copy DataFrame     : 1.8s
Cast DPD           : 0.1s
Groupby            : 0.1s
Colonnes de travail: 0.9s
[LGD] Loans exclus pour EAD=0 (artefact de séquence) : 196
[LGD] Loans avec target calculée : 30807 / 30807
[LGD] Observations clippées hors [0,1] : 4238
[LGD] Distribution LGD :
count    30807.0000
mean         0.3668
std          0.2918
min          0.0000
25%          0.1260
50%          0.3288
75%          0.5417
max          1.0000
Name: lgd_target, dtype: float64
🏃 View run LGD_scaler_preprocessing_v1 at: http://localhost:5000/#/experiments/3/runs/2037a443f35748359cc6d6d744490efa
🧪 View experiment at: http://localhost:5000/#/experiments/3
datas scaled


In [10]:
X.shape

(30807, 20)

In [11]:
y.shape

(30807,)

# Split and save

In [13]:
from sklearn.model_selection import train_test_split

# 1. Separate the full dataset into Training (80%) and a temporary Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Split the temp set into Training (70%) and Validation (30% of the temp set, or 15% of the total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.30, random_state=42)

In [14]:
#X_train.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_train.csv',sep=',',index=False)
#X_val.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_val.csv',sep=',',index=False)
#X_test.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_test.csv',sep=',',index=False)

In [15]:
#pd.DataFrame(y_train).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_train.csv',sep=',',index=False)
#pd.DataFrame(y_val).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_val.csv',sep=',',index=False)
#pd.DataFrame(y_test).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_test.csv',sep=',',index=False)#

# Run

train

In [16]:

import src.LGDcomponent.run.lgbm_multiclass as lgbm
importlib.reload(lgbm)

from src.LGDcomponent.run.lgbm_multiclass import LgbmMulticlass as LGBM_Multiclass

In [17]:
train_map = {'x_train':X_train, 'y_train':y_train}
val_map = {'x_val':X_val, 'y_val':y_val}

In [18]:
train = LGBM_Multiclass(train_map = train_map, val_map = val_map, config_path = train_cfPath,test_path = test_cfPath)

In [19]:
train.run()

[I 2026-07-02 03:16:45,932] A new study created in memory with name: no-name-357dee86-b4b6-4980-908a-3653851bdb19


[LGDDiscretizer] n_bins ajusté : 8 demandés →  7 bins effectifs (doublons dans la distribution).


[I 2026-07-02 03:16:49,126] Trial 0 finished with value: -0.26959153403012714 and parameters: {'max_depth': 3, 'num_leaves': 61, 'min_child_samples': 25, 'reg_lambda': 5.1547136684282115, 'reg_alpha': 0.9924704135211533, 'subsample': 0.6730081358634735, 'colsample_bytree': 0.6500869619997532, 'learning_rate': 0.022413157840355205, 'n_estimators': 459}. Best is trial 0 with value: -0.26959153403012714.


RMSE: 0.2696 | Dxy: 0.2924 | ECE: 0.0168


[I 2026-07-02 03:16:50,945] Trial 1 finished with value: -0.26811661962432864 and parameters: {'max_depth': 5, 'num_leaves': 44, 'min_child_samples': 44, 'reg_lambda': 3.0902421735300276, 'reg_alpha': 0.7725372478114543, 'subsample': 0.7177138601473081, 'colsample_bytree': 0.912878060992401, 'learning_rate': 0.034614852480025014, 'n_estimators': 294}. Best is trial 0 with value: -0.26959153403012714.


...

RMSE: 0.2691 | Dxy: 0.2960 | ECE: 0.0197


[I 2026-07-02 03:17:11,843] Trial 18 finished with value: -0.27127496312861493 and parameters: {'max_depth': 3, 'num_leaves': 53, 'min_child_samples': 21, 'reg_lambda': 7.632034733239695, 'reg_alpha': 0.2709542934038678, 'subsample': 0.9607685970065779, 'colsample_bytree': 0.6041020426986916, 'learning_rate': 0.02064522144384006, 'n_estimators': 290}. Best is trial 13 with value: -0.2763284610663373.


RMSE: 0.2713 | Dxy: 0.2854 | ECE: 0.0191


[I 2026-07-02 03:17:13,494] Trial 19 finished with value: -0.2676400370957944 and parameters: {'max_depth': 6, 'num_leaves': 80, 'min_child_samples': 33, 'reg_lambda': 5.3078932520562905, 'reg_alpha': 0.4950830243883838, 'subsample': 0.833992986823924, 'colsample_bytree': 0.7169507389423984, 'learning_rate': 0.050090056562962414, 'n_estimators': 235}. Best is trial 13 with value: -0.2763284610663373.


RMSE: 0.2676 | Dxy: 0.3032 | ECE: 0.0146
🏃 View run LightGBM LGD Multiclass Train at: http://localhost:5000/#/experiments/3/runs/5d1f5484377d453a97625d3c424ffdde
🧪 View experiment at: http://localhost:5000/#/experiments/3


test

In [20]:
bin_edges = train.discretizer.bin_edges_

In [21]:
bin_edges

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

In [22]:
test_map ={'x_test':X_test, 'y_test':y_test}

In [23]:
test = LGBM_Multiclass(test_map=test_map, config_path = test_cfPath)

In [24]:
test.run()

🏃 View run LightGBM LGD Multiclass Test at: http://localhost:5000/#/experiments/3/runs/bebf0b76492d4380a893ca19e3f7ab59
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [25]:
test_discretizer = test.discretizer

In [26]:
test_discretizer.bin_edges_

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

# compute lgd

In [ ]:
#midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
# [0.0623, 0.1778, 0.2788, 0.3775, 0.4832, 0.6291, 0.8600]

#y_predict = sum(proba[0] * midpoints)
# = 0.1673*0.0623 + 0.0426*0.1778 + 0.0483*0.2788 + 0.0613*0.3775
#   + 0.0533*0.4832 + 0.0908*0.6291 + 0.5363*0.8600
# ≈ 0.587

# Test inference

In [27]:
import src.LGDcomponent.LgdPrediction as lgd
importlib.reload(lgd)
from src.LGDcomponent.LgdPrediction import LGDPrediction

In [28]:
mlflow_config ='/Users/macbookpro/platform/Backend/configs/Lgd_mlFlow_config.yaml'
model_config = '/Users/macbookpro/platform/Backend/configs/Lgd_model_config.yaml'

In [29]:
inference = LGDPrediction(hist=loan_hist,orig=loan_orig,mlflow_config=mlflow_config, model_config=model_config)

Copy DataFrame     : 0.0s
Cast DPD           : 0.0s
Groupby            : 0.0s
Colonnes de travail: 0.0s


In [30]:
discretize = inference.discretizer

In [31]:
type(discretize)

pipelines.Features.Lgd_discretizer.LGDDiscretizer

In [32]:
inference.apply()

[np.float64(0.4757785148901206)]

choisir des profile, et stocker toute leur performance et origination dans le warehouse, pour le Pd, on construira leur windows .

In [ ]:
# window et sans window ( preselection des profile sans window) dans le warehouse,